In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j

import os
import sys

import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,190 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,523 kB]
Get:13 http://security.ubuntu.com/ubuntu

In [2]:
spark = SparkSession.builder.appName("Mi primera").getOrCreate()

In [3]:
!apt-get install git

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.11).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.


In [4]:
repository_url = "https://github.com/fruits-360/fruits-360-100x100"  # Replace with your repo URL
!git clone {repository_url}

Cloning into 'fruits-360-100x100'...
remote: Enumerating objects: 94555, done.
remote: Counting objects: 100% (3809/3809), done.
remote: Compressing objects: 100% (3799/3799), done.
remote: Total 94555 (delta 32), reused 3786 (delta 10), pack-reused 90746 (from 1)
Receiving objects: 100% (94555/94555), 723.85 MiB | 31.26 MiB/s, done.
Resolving deltas: 100% (33/33), done.
Updating files: 100% (94112/94112), done.


Se genera un rdd para las imágenes de la carpeta Training del repositorio de Fruits-360.

In [5]:
import os
from pyspark.sql import SparkSession

# Assuming images are in the 'images' directory within 'fruits-360-100x100'
repo_name = "fruits-360-100x100"  # Replace with your repository folder name

# Fix: Use the repository name directly as the path
# os.path.abspath() is not needed here and was causing the issue
repo_path = "/content/fruits-360-100x100"
repo_training_path = "/content/fruits-360-100x100/Training"
repo_test_path = "/content/fruits-360-100x100/Test"

# Change to the repository directory
os.chdir(repo_path) #Change the current working directory

# Get list of image training file paths
# Use 'in' instead of 'contains' to check if substring is present in filename


import random

# Define the folder names to include

def get_direct_subfolders(path):
    """
    Get a list of folder names directly inside the specified path.

    Args:
        path (str): Path to the directory.

    Returns:
        list: List of folder names.
    """
    try:
        subfolders = [folder for folder in os.listdir(path) if os.path.isdir(os.path.join(path, folder))]
        return subfolders
    except Exception as e:
        print(f"Error accessing the path {path}: {e}")
        return []

folder_names = get_direct_subfolders(repo_training_path)

# Dictionary to store selected files for each folder
selected_files_by_folder = {}



# Iterate over each folder
for folder_name in folder_names:
    # Collect all files in the current folder
    folder_files = [
        os.path.join(root, f)
        for root, dirs, files in os.walk(repo_training_path)
        if folder_name in root
        for f in files if f.endswith(('.png', '.jpg', '.jpeg'))
    ]
    # Randomly select up to 100 files from the folder
    selected_files_by_folder[folder_name] = random.sample(folder_files, min(10, len(folder_files)))

# Combine all selected files into one list if needed
image_training_files = [file for files in selected_files_by_folder.values() for file in files]



# Create an RDD from the list
image_rdd = spark.sparkContext.parallelize(image_training_files)

Realizar alguna operación en el RDD, como estadísticas descriptivas básicas.

In [7]:
# Contar imágenes

image_count = image_rdd.count()
print(f"Número total de imágenes: {image_count}")


Número total de imágenes: 1410


In [8]:
# Se calcula estadísticas básicas con el tamaño de archivo de las imágenes.

# Map file paths to their sizes
file_sizes_rdd = image_rdd.map(lambda file: os.path.getsize(file))

# Compute basic statistics
file_sizes_stats = file_sizes_rdd.stats()
print(f"Estadísticas sobre el tamaño de archivo de las imágenes: {file_sizes_stats}")


Estadísticas sobre el tamaño de archivo de las imágenes: (count: 1410, mean: 4422.002836879434, stdev: 1071.9355080065611, max: 7398.0, min: 1099.0)


In [9]:
# Se obtiene un histograma de los tamaños de archivo.

# Use Spark's histogram function
file_size_histogram = file_sizes_rdd.histogram(10)  # Adjust bins as needed
print(f"Histograma del tamaño de archivo de las imágenes: {file_size_histogram}")


Histograma del tamaño de archivo de las imágenes: ([1099.0, 1728.9, 2358.8, 2988.7, 3618.6, 4248.5, 4878.4, 5508.3, 6138.2, 6768.099999999999, 7398], [60, 27, 41, 62, 305, 478, 268, 120, 35, 14])
